# EP1 - Limpieza inicial (Colab / Jupyter)

Este notebook ejecuta el proceso de limpieza inicial y genera: `data/processed/movies_processed.csv`, `data/processed/tv_processed.csv` y `documentation/markdown/EP1_data_cleaning.md`.

Instrucciones rápidas:
- En Colab: sube los dos CSV (`netflix_movies_detailed_up_to_2025.csv` y `netflix_tv_shows_detailed_up_to_2025.csv`) usando el botón 
 o el siguiente comando de subida.
- En Jupyter local: asegúrate de ejecutar el notebook desde la raíz del repo (donde está la carpeta `data/`).

In [8]:
    "# Detectar Colab: si está en Colab, pedir subida; si no, usar rutas locales Windows.",
    "movies_path = None",
    "tv_path = None",
    "try:",
    "    from google.colab import files\n    ",
    "    print('Entorno Colab detectado. Usa el cuadro de diálogo para subir los CSVs.')",
    "    uploaded = files.upload()",
    "    for k in uploaded.keys():",
    "        name = k.lower()",
    "        if 'movie' in name and movies_path is None:",
    "            movies_path = k",
    "        if ('tv' in name or 'show' in name) and tv_path is None:",
    "            tv_path = k",
    "    print('movies_path=', movies_path, 'tv_path=', tv_path)",
    "except Exception:",
    "    # No Colab: usar las rutas locales en Windows (ruta provista por el usuario).",
    "    movies_path = r'C:\\Users\\shein\\OneDrive\\Documentos\\GitHub\\visualizacion-de-datos-StreamView-Analytics\\data\\netflix_movies_detailed_up_to_2025.csv'",
    "    tv_path = r'C:\\Users\\shein\\OneDrive\\Documentos\\GitHub\\visualizacion-de-datos-StreamView-Analytics\\data\\netflix_tv_shows_detailed_up_to_2025.csv'",
    "    import os",
    "    print('Usando rutas locales:')",
    "    print('movies_path=', movies_path)",
    "    print('tv_path=', tv_path)",
    "    # Comprobación rápida de existencia de archivos",
    "    if not os.path.exists(movies_path):",
    "        print('Warning: movies file not found at', movies_path)",
    "    if not os.path.exists(tv_path):",
    "        print('Warning: tv file not found at', tv_path)"

"        print('Warning: tv file not found at', tv_path)"

In [7]:
import os
import pandas as pd
from datetime import datetime
ROOT = os.getcwd()
processed_dir = os.path.join(ROOT, 'data', 'processed')
docs_dir = os.path.join(ROOT, 'documentation', 'markdown')
os.makedirs(processed_dir, exist_ok=True)
os.makedirs(docs_dir, exist_ok=True)
files = [('movies', movies_path), ('tv', tv_path)]
summary = {'run_date': datetime.utcnow().isoformat() + 'Z', 'files': {}}
for key, path in files:
    info = {'path': path}
    if not os.path.exists(path):
        info['error'] = 'file not found'
        summary['files'][key] = info
        continue
    df = pd.read_csv(path, low_memory=False)
    info['original_rows'] = int(len(df))
    full_dup = int(df.duplicated().sum())
    info['full_row_duplicates'] = full_dup
    if 'show_id' in df.columns:
        id_dup = int(df.duplicated(subset=['show_id']).sum())
        info['show_id_duplicates'] = id_dup
    else:
        info['show_id_duplicates'] = None
    df = df.drop_duplicates()
    info['rows_after_dedup'] = int(len(df))
    obj_cols = df.select_dtypes(include=['object']).columns.tolist()
    for c in obj_cols:
        try:
            df[c] = df[c].astype(str).str.strip()
            df.loc[df[c] == 'nan', c] = pd.NA
        except Exception:
            pass
    if 'date_added' in df.columns:
        info['date_added_parsed_nulls_before'] = int(df['date_added'].isnull().sum())
        df['date_added'] = pd.to_datetime(df['date_added'], errors='coerce')
        info['date_added_parsed_nulls_after'] = int(df['date_added'].isnull().sum())
    if 'country' in df.columns:
        df['country'] = df['country'].where(df['country'].isna(), df['country'].str.title())
    df = df.replace({'': pd.NA, 'None': pd.NA, 'nan': pd.NA})
    null_counts = df.isnull().sum().sort_values(ascending=False)
    info['null_counts_top10'] = null_counts.head(10).to_dict()
    info['total_nulls'] = int(null_counts.sum())
    out_path = os.path.join(processed_dir, f'{key}_processed.csv')
    df.to_csv(out_path, index=False)
    info['processed_path'] = out_path
    info['processed_rows'] = int(len(df))
    summary['files'][key] = info
print('Procesamiento completo. Archivos guardados en data/processed/')

NameError: name 'movies_path' is not defined

In [ ]:
# Escribir reporte MD con resumen de la limpieza
md_path = os.path.join(docs_dir, 'EP1_data_cleaning.md')
with open(md_path, 'w', encoding='utf-8') as f:
    f.write('---\n')
    f.write("title: "EP1 — Data cleaning inicial"\n")
    f.write("author: "Equipo StreamView (notebook)"\n")
    f.write(f"date: {datetime.utcnow().date()}\n")
    f.write('"source_files:\n")
    for _, p in files:
        f.write(f"  - {p}\n")
    f.write('---\n\n')
    f.write('# Resumen de la limpieza inicial\n\n')
    f.write(f'Fecha de ejecución (UTC): {summary["run_date"]}\n\n')
    for key, info in summary['files'].items():
        f.write(f'## Archivo: {key}\n\n')
        if 'error' in info:
            f.write(f'- Error: {info["error"]}\n\n')
            continue
        f.write(f'- Ruta original: {info["path"]}\n')
        f.write(f'- Filas originales: {info.get("original_rows")}\n')
        f.write(f'- Filas después de eliminar duplicados: {info.get("rows_after_dedup")}\n')
        f.write(f'- Duplicados (filas completas): {info.get("full_row_duplicates")}\n')
        if info.get('show_id_duplicates') is not None:
            f.write(f'- Duplicados por `show_id`: {info.get("show_id_duplicates")}\n')
        if 'date_added_parsed_nulls_before' in info:
            f.write(f'- `date_added` nulos antes: {info.get("date_added_parsed_nulls_before")}\n')
            f.write(f'- `date_added` nulos después de parseo: {info.get("date_added_parsed_nulls_after")}\n')
        f.write(f'- Filas procesadas guardadas en: {info.get("processed_path")}\n')
        f.write(f'- Total de valores nulos (suma por columnas): {info.get("total_nulls")}\n')
        f.write('\n')
        f.write('### Top 10 columnas por valores faltantes\n\n')
        f.write('| Columna | Nulos |\n')
        f.write('|---|---:|\n')
        for col, n in info.get('null_counts_top10', {}).items():
            f.write(f'| {col} | {n} |\n')
        f.write('\n')
print(f'Reporte escrito en: {md_path}')